# U03-A05 — Oficina — Primeira base processável

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_03/04_oficina_base_processavel.ipynb)

Este notebook é um roteiro de projeto. Execute transformações em uma cópia
própria ou notebook técnico; registre aqui decisões, evidências e resultados.
Não altere os arquivos brutos.

![Dados brutos, código, dados derivados e documentação formam um pacote submetido a testes e parecer de qualidade, com retorno rastreável para correções.](imagens/04_pacote_processavel.svg)

A figura funciona também como checklist: um CSV isolado não demonstra como a
base foi produzida, testada ou delimitada. A entrega precisa combinar os
componentes e manter vínculos entre eles.

A oficina começa pelo inventário porque nenhuma transformação pode ser
avaliada sem conhecer pergunta, unidade, entradas, versões e política de
preservação. Essa descrição estabelece o escopo de todas as etapas seguintes.

## 1. Inventário e estrutura

**Pergunta e unidade de análise:**

Como a palavra "liberdade" é empregada e ressignificada nos discursos de posse presidencial no Brasil entre 1985 e 2023? A unidade de análise central é o parágrafo/trecho de discurso (trecho_id) contendo ocorrências explícitas do termo e seu contexto sintático/semântico imediato.

**Arquivos, formatos, versões e proveniência:**

Corpus composto por 12 arquivos de texto nativo em formato .txt, contendo as transcrições integrais dos discursos de posse presidencial da Redemocratização (1985–2023) obtidos de portais oficiais públicos (Biblioteca da Presidência e Congresso Nacional). Acompanhado da tabela de metadados dos discursos em .csv e do arquivo de rastreabilidade proveniencia_pipeline.json.

**Estrutura de pastas e política de imutabilidade:**

- dados/brutos/: Arquivos .txt e metadados originais.

- dados/intermediarios/: Tabela de catálogo normalizado e logs de transformações.

- dados/derivados/: Tabelas consolidadas finalizadas (documentos_processaveis.csv e discursos_trechos.csv) destinadas à análise.

- Política rigorosa de não sobrescrita: nenhum dado bruto é alterado durante a execução dos scripts.

**Ambiente e dependências:**

Ambiente Python 3.x (Local / Google Colab) com uso das bibliotecas nativas pathlib, json, re, unicodedata e da biblioteca pandas.

Com as entradas e o ambiente identificados, podemos registrar como cada fonte
será convertida em uma representação legível. As escolhas de importação e
extração precisam produzir saídas compatíveis com o inventário declarado.

## 2. Importação e extração

| Fonte | Leitor/parâmetros | Estrutura esperada | Teste | Saída |
|---|---|---|---|---|
| Discursos de Posse (`.txt`) | `pathlib.Path.read_text(encoding="utf-8")` | Texto corrido não estruturado (parágrafos por `\n\n`) | Teste de não-vazio (`len > 0`) e detecção do termo `r"\bliberdades?\b"` | Strings de texto bruto com enquadramento UTF-8 |
| Tabela de Metadados (`.csv`) | `pd.read_csv("...", sep=",", encoding="utf-8")` | Tabela bidimensional plana (linhas/colunas) | Verificação de chave única `id_doc` sem nulos | DataFrame Pandas intermediário |

**PDFs com texto, PDFs de imagem e TXT:**

O projeto utiliza exclusivamente textos digitais nativos em .txt. Não há utilização de arquivos PDF (textuais ou escaneados) no corpus.

**Plano de OCR, amostra de controle e métrica de erro:**

OCR desnecessário (0% de necessidade). Os discursos foram obtidos diretamente em texto corrido limpo, dispensando etapas de visão computacional, ferramentas de transcrição de imagem ou métricas de erro como CER/WER.

A importação produz objetos técnicos, mas ainda não garante uma organização
coerente para análise. O modelo tabular define o que cada linha representa e
quais transformações serão aplicadas sem apagar os valores recebidos.

## 3. Modelo tabular e transformação

**Tabelas e unidade de cada linha:**

- documentos_processaveis.csv: Unidade = um discurso de posse presidencial integral (id_doc).

- discursos_trechos.csv: Unidade = um parágrafo contendo a palavra "liberdade" (trecho_id).

- motivos_exclusao.csv: Unidade = um pronunciamento/documento descartado.

**Decisão largo/longo:**

A tabela de trechos adota a estrutura longa, onde cada linha representa uma relação única entre um trecho (trecho_id), o discurso ao qual pertence (id_doc), a ocorrência lexical e os metadados associados ao presidente/ano.

| Campo | Original preservado | Regra | Justificativa | Teste | Perda possível |
|---|---|---|---|---|---|
| `texto_conteudo` | `texto_bruto` | Normalização de espaços e remoção de quebras de linha falsas via regex (`UTF-8`) | Garantir segmentação precisa por parágrafos | Inspeção visual de amostra e validação de encoding sem *mojibake* | Apaga marcadores de hifenização de edições impressas |
| `id_doc` | `nome_arquivo` | Padronização no formato `DISCURSO_ANO_PRESIDENTE` | Criar chave primária única persistente | Teste `.is_unique` em Pandas | Oculta a denominação de arquivo original do custodiante |
| `presidente` / `partido` | `presidente_original` | Mapeamento explícito para nomes políticos e siglas oficiais | Assegurar agregação exata sem duplicação de grafias | Verificação via `.value_counts()` e checagem de nulos | Suprime variações de grafia ou nomes informais |
| `trecho_id` | N/A (campo derivado) | Segmentação por parágrafo e acréscimo do sufixo `_PAR_N` | Criar a unidade de análise do parágrafo | Validação da chave primária do trecho (`is_unique`) | Fragmenta o fluxo contínuo da oratória do discurso |

As regras do modelo tabular tornam valores comparáveis e auditáveis. Essa
comparabilidade permite diagnosticar ausências e semelhanças, mas não autoriza
preencher lacunas ou eliminar registros automaticamente.

## 4. Ausências e duplicatas

**Representações de ausência e razões:**

Ocorrências ausentes de valores são tratadas com pd.NA em colunas de texto/categorias.

Para ausências com significado (ex.: pronunciamento sem menção ao termo "liberdade"), cria-se a coluna razao_ausencia indicando a justificativa.

**Chave de duplicata exata:**

A chave primária id_doc identifica cada discurso de posse de forma única. Na tabela derivada de trechos, a chave primária é trecho_id.

**Critérios de possível duplicata e revisão humana:**

Identificação de parágrafos idênticos extraídos do mesmo id_doc ou discursos duplicados para o mesmo ano/presidente por meio da checagem em Pandas:

df.duplicated(subset=["id_doc", "texto_trecho"], keep=False)

**Decisão de manter, relacionar, fundir ou excluir:**

Documentos ou parágrafos confirmados como duplicados são descartados da análise, e o evento é obrigatoriamente documentado na tabela motivos_exclusao.csv com ID, data e justificativa.

Depois de representar ausências e revisar possíveis duplicatas, as unidades
estão mais bem definidas para integração. Agora é necessário declarar como as
tabelas se relacionam e testar se a junção preserva o número e a identidade dos casos.

## 5. Junções e integração

| Tabelas | Chave | Cardinalidade | Validação | Não correspondências |
|---|---|---|---|---|
| `documentos_processaveis` → `discursos_trechos` | `id_doc` | `1:N` | `validate="one_to_many"` no Pandas | Rastreadas via `indicator=True`, discursos sem trechos retornam `left_only` sem descarte do metadado |

**Vínculo entre texto, metadado, indicador e fonte:**

Toda linha da tabela derivada discursos_trechos.csv retém obrigatoriamente o campo id_doc como chave estrangeira, com o objetivo de rastrear qualquer parágrafo de volta ao seu discurso bruto de origem em .txt e ao seu registro de proveniência em JSON.

**Contagens antes/depois e riscos de multiplicação:**

A tabela central contém exatamente 12 discursos. A junção/extração para a tabela de trechos expande intencionalmente esse volume para ~60 a 120 registros (parágrafos). O risco de multiplicação indevida de documentos é controlado testando df["id_doc"].nunique() == 12 antes e depois das operações.

As junções produzem a base integrada, mas o resultado ainda precisa demonstrar
como foi construído. Logs e testes ligam cada saída às entradas, parâmetros e
decisões que tornam o processo reproduzível.

## 6. Registro e testes

**Log de transformações:**

Mantido no arquivo dados/intermediarios/log_transformacoes.csv, registrando para cada campo: problema, regra aplicada, justificativa, linhas afetadas, teste executado, reversibilidade e responsável.

**Testes de esquema, domínios, chaves e arquivos:**

- Teste de Unicidade: assert base_documentos["id_doc"].is_unique.

- Teste de Integridade Referencial: assert set(discursos_trechos["id_doc"]).issubset(set(base_documentos["id_doc"])).

- Teste de Não-Nulo: verificação da ausência de valores nulos nas colunas id_doc, ano e presidente.

**Como reconstruir a base desde os brutos:**

Executando sequencialmente os notebooks do projeto a partir da pasta raiz. O script lê os .txt em dados/brutos/, aplica as regras de limpeza, gera o log em dados/intermediarios/ e exporta os datasets finalizados em dados/derivados/ de maneira 100% automatizada e sem intervenção manual.

**Erros conhecidos e casos pendentes:**

Fragmentação de frases no início ou fim de parágrafos e variação de termos no plural ("liberdades"), tratados no filtro regex com a borda de palavra r"\bliberdades?\b".

Com inventário, transformações, relações e testes documentados, já podemos
avaliar o pacote como um todo. A rubrica converte os requisitos metodológicos
em critérios comuns para autoavaliação.

## 7. Rubrica

| Critério | 0 | 1 | 2 | Minha nota/evidência |
|---|---|---|---|---|
| **preservação e proveniência** | ausente | parcial | auditável | **2** — Arquivos brutos `.txt` imutáveis e proveniência documentada em JSON. |
| **importação** | implícita | parâmetros parciais | testada | **2** — Leitura explícita com encoding UTF-8 nativo e testes de validação. |
| **limpeza** | sobrescreve | regras incompletas | original + regra + teste | **2** — Regras registradas no log, mantendo o dado original rastreável. |
| **ausências/duplicatas** | descartadas | diagnosticadas | justificadas | **2** — Ausências justificadas e descartes registrados em `motivos_exclusao.csv`. |
| **junções** | sem controle | chave declarada | cardinalidade auditada | **1** — Cardinalidade `1:N` validada no Pandas com parâmetro `indicator`. |
| **integração** | arquivos soltos | vínculos parciais | chaves verificadas | **1** — Tabela central ligada aos trechos via chave estrangeira `id_doc`. |
| **reprodutibilidade** | manual | ordem parcial | reconstruível | **1** — Execução do pipeline totalmente automatizada via código em Python. |

A autoavaliação identifica lacunas segundo critérios conhecidos. A revisão por
pares acrescenta uma tentativa externa de reconstruir decisões, localizar
junções perigosas e verificar limites que o autor pode não ter percebido.

## U03-A06 — 8. Entrega e revisão por pares

Entregue inventário, notebooks, base processável, tabelas relacionais, log,
relatório de qualidade e ficha de proveniência. O colega deve tentar localizar
uma regra não reversível, uma junção perigosa e um limite não documentado.

**Parecer recebido e mudanças:** Escreva aqui.

**O que a Unidade 4 poderá explorar e o que ainda não deve concluir:**
Escreva aqui. Esta resposta encerra a Unidade 3: ela distingue a base que já
pode ser explorada das questões de qualidade que ainda condicionam qualquer
descrição, gráfico ou hipótese na unidade seguinte.